# Amazon ML Challenge 2026 — Day 1: EDA & Baseline Pipeline

**Official Evaluation Metric:** Macro-averaged $F_{0.5}$ (Precision-weighted entity resolution: $\beta=0.5$)

### Day 1 Scope
1. Load datasets safely (`data/dataset/train/`)
2. Shape and column verification
3. Missing value analysis
4. Country distribution analysis
5. Duplicate record analysis
6. Ground truth match count distribution (singletons vs multi-matches)
7. Inspect real matched entity pairs (name & address noise patterns)
8. Validation split design
9. Fast baseline pipeline (Exact normalized & high-precision token match)
10. Validation score ($F_{0.5}$, Precision, Recall)
11. Generate valid test submission files (`matching_results.tsv`, `candidate_pairs.tsv`)
12. Run official format validator (`utils/validate_submission.py`)

In [ ]:
import os
import sys
import re
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Find dataset directory dynamically
POSSIBLE_DIRS = ['../data/dataset/train', 'data/dataset/train', '../dataset/train', 'dataset/train']
TRAIN_DIR = next((d for d in POSSIBLE_DIRS if os.path.exists(d)), None)
if not TRAIN_DIR:
    raise FileNotFoundError('Could not locate train dataset directory.')

POSSIBLE_TEST_DIRS = ['../data/dataset/test', 'data/dataset/test', '../dataset/test', 'dataset/test']
TEST_DIR = next((d for d in POSSIBLE_TEST_DIRS if os.path.exists(d)), None)

print(f'Train directory located: {os.path.abspath(TRAIN_DIR)}')
print(f'Test directory located : {os.path.abspath(TEST_DIR)}')
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('src'))

## 1. Quick Inspection with Sample (Controlled Memory)
To inspect schemas and avoid unneeded memory usage, we first inspect sample rows with `nrows=1000`.

In [ ]:
s1_sample = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source1.tsv'), sep='\t', nrows=1000)
s2_sample = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source2.tsv'), sep='\t', nrows=1000)
s3_sample = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source3.tsv'), sep='\t', nrows=1000)
gt_sample = pd.read_csv(os.path.join(TRAIN_DIR, 'train_ground_truth.tsv'), sep='\t', nrows=1000)

print('Source 1 Columns:', s1_sample.columns.tolist())
print('Source 2 Columns:', s2_sample.columns.tolist())
print('Source 3 Columns:', s3_sample.columns.tolist())
print('Ground Truth Columns:', gt_sample.columns.tolist())
print('\n=== S1 Sample ===')
print(s1_sample.head(3))
print('\n=== Ground Truth Sample ===')
print(gt_sample.head(3))

## 2. Load Full Dataset with Optimized Dtypes

In [ ]:
# Load with memory-efficient string/categorical dtypes
dtype_spec = {
    'entity_id': 'string',
    'business_name': 'string',
    'business_address': 'string',
    'country': 'category'
}

print('Loading Source 1...')
s1 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source1.tsv'), sep='\t', dtype=dtype_spec)
print(f'S1 Loaded: {s1.shape[0]:,} rows')

print('Loading Ground Truth...')
gt = pd.read_csv(os.path.join(TRAIN_DIR, 'train_ground_truth.tsv'), sep='\t', dtype={'source1_entity_id': 'string', 'matched_entity_ids': 'string'})
print(f'Ground Truth Loaded: {gt.shape[0]:,} rows')

# S2 and S3 are ~5M rows each
print('Loading Source 2...')
s2 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source2.tsv'), sep='\t', dtype=dtype_spec)
print(f'S2 Loaded: {s2.shape[0]:,} rows')

print('Loading Source 3...')
s3 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source3.tsv'), sep='\t', dtype=dtype_spec)
print(f'S3 Loaded: {s3.shape[0]:,} rows')

## 3. Data Integrity & Missing Values Analysis

In [ ]:
for name, df in [('Source 1', s1), ('Source 2', s2), ('Source 3', s3)]:
    missing = df.isnull().sum()
    pct = (df.isnull().mean() * 100).round(2)
    print(f'=== {name} Missing Values ===')
    print(pd.DataFrame({'Missing': missing, 'Percent (%)': pct}))
    print()

## 4. Country Distribution Analysis
Checking geographic distribution across all sources.

In [ ]:
for name, df in [('Source 1', s1), ('Source 2', s2), ('Source 3', s3)]:
    print(f'=== {name} Top Countries ===')
    print(df['country'].value_counts(dropna=False).head(10))
    print()

## 5. Duplicate Records & Entity Key Analysis

In [ ]:
print(f'S1 Unique Entity IDs: {s1["entity_id"].nunique():,} / {len(s1):,}')
print(f'S2 Unique Entity IDs: {s2["entity_id"].nunique():,} / {len(s2):,}')
print(f'S3 Unique Entity IDs: {s3["entity_id"].nunique():,} / {len(s3):,}')
print(f'GT Unique S1 IDs    : {gt["source1_entity_id"].nunique():,} / {len(gt):,}')

## 6. Ground Truth Match Distribution Analysis
Understanding singletons (zero matches) vs multi-match records.

In [ ]:
def parse_matches(val):
    if pd.isna(val) or str(val).strip() == '':
        return []
    return [m.strip() for m in str(val).split(',') if m.strip()]

gt['match_list'] = gt['matched_entity_ids'].apply(parse_matches)
gt['num_matches'] = gt['match_list'].apply(len)

match_counts = gt['num_matches'].value_counts().sort_index()
print('Match count distribution per S1 entity:')
print(match_counts)

zero_matches = (gt['num_matches'] == 0).sum()
total = len(gt)
print(f'\nSingletons (0 matches): {zero_matches:,} ({zero_matches / total * 100:.2f}%)')
print(f'Entities with >=1 match: {total - zero_matches:,} ({(total - zero_matches) / total * 100:.2f}%)')
print(f'Max matches for a single S1 entity: {gt["num_matches"].max()}')

## 7. Inspect Real Matched Pairs (Noise Patterns in Name & Address)

In [ ]:
s1_idx = s1.set_index('entity_id')
s2_idx = s2.set_index('entity_id')
s3_idx = s3.set_index('entity_id')

sample_matched = gt[gt['num_matches'] > 0].head(5)
for _, row in sample_matched.iterrows():
    s1_id = row['source1_entity_id']
    s1_row = s1_idx.loc[s1_id]
    print(f'=== Source 1 [{s1_id}] ===')
    print(f'  Name   : {s1_row["business_name"]}')
    print(f'  Address: {s1_row["business_address"]}')
    print(f'  Country: {s1_row["country"]}')
    for mid in row['match_list']:
        target_idx = s2_idx if mid.startswith('S2-') else s3_idx
        if mid in target_idx.index:
            m_row = target_idx.loc[mid]
            print(f'  -> Matched [{mid}]:')
            print(f'       Name   : {m_row["business_name"]}')
            print(f'       Address: {m_row["business_address"]}')
            print(f'       Country: {m_row["country"]}')
    print('-' * 80)

## 8. Validation Split (S1 Entity Split)
We split Source 1 into 95% train and 5% validation. Because matching is evaluated per Source 1 entity, splitting on `source1_entity_id` prevents data leakage.

In [ ]:
np.random.seed(42)
all_s1_ids = gt['source1_entity_id'].values
n_val = min(50000, int(len(all_s1_ids) * 0.05))
val_ids = set(np.random.choice(all_s1_ids, size=n_val, replace=False))

val_gt = gt[gt['source1_entity_id'].isin(val_ids)].copy()
print(f'Validation set size: {len(val_gt):,} S1 entities')

## 9. Baseline Pipeline (Normalized Match + High-Precision Token Overlap)
A fast Day-1 baseline linking entities with matching normalized name and country.

In [ ]:
def clean_name(s):
    if pd.isna(s): return ''
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\b(inc|corp|ltd|llc|gmbh|co|company)\b', '', s)
    return ' '.join(s.split())

print('Normalizing validation sample names...')
val_s1 = s1[s1['entity_id'].isin(val_ids)].copy()
val_s1['norm_name'] = val_s1['business_name'].apply(clean_name)

# Build index on candidates (S2 and S3)
print('Indexing candidates...')
cand_df = pd.concat([
    s2[['entity_id', 'business_name', 'country']],
    s3[['entity_id', 'business_name', 'country']]
], ignore_index=True)
cand_df['norm_name'] = cand_df['business_name'].apply(clean_name)

# Map (country, norm_name) -> list of entity_ids
cand_index = defaultdict(list)
for cid, norm, ctry in zip(cand_df['entity_id'], cand_df['norm_name'], cand_df['country']):
    if norm:
        cand_index[(str(ctry), norm)].append(cid)

print(f'Candidate index populated with {len(cand_index):,} unique keys.')

## 10. Compute Baseline Predictions & Macro $F_{0.5}$ Evaluation

In [ ]:
val_preds = {}
for sid, norm, ctry in zip(val_s1['entity_id'], val_s1['norm_name'], val_s1['country']):
    matches = cand_index.get((str(ctry), norm), [])
    val_preds[sid] = set(matches[:5]) # cap to top 5

# Evaluate official Macro F0.5
beta = 0.5
f_scores = []
precisions = []
recalls = []

for _, row in val_gt.iterrows():
    sid = row['source1_entity_id']
    true_set = set(row['match_list'])
    pred_set = val_preds.get(sid, set())
    
    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    
    if len(true_set) == 0 and len(pred_set) == 0:
        f_scores.append(1.0)
        precisions.append(1.0)
        recalls.append(1.0)
    elif len(pred_set) == 0 or len(true_set) == 0:
        f_scores.append(0.0)
        precisions.append(0.0 if len(pred_set) > 0 else 1.0)
        recalls.append(0.0 if len(true_set) > 0 else 1.0)
    else:
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f05 = ((1 + beta**2) * p * r) / ((beta**2 * p) + r) if ((beta**2 * p) + r) > 0 else 0.0
        f_scores.append(f05)
        precisions.append(p)
        recalls.append(r)

print('=== Day 1 Baseline Validation Results ===')
print(f'Macro F0.5   : {np.mean(f_scores):.4f}')
print(f'Macro Precision: {np.mean(precisions):.4f}')
print(f'Macro Recall   : {np.mean(recalls):.4f}')

## 11. Generate Baseline Test Submission & Candidate Pairs

In [ ]:
os.makedirs('../output', exist_ok=True)
os.makedirs('output', exist_ok=True)
out_dir = '../output' if os.path.exists('../output') else 'output'

matching_path = os.path.join(out_dir, 'matching_results.tsv')
candidate_path = os.path.join(out_dir, 'candidate_pairs.tsv')

print(f'Generating submissions into: {out_dir}')

# Read test source 1 to ensure 100% of test entities are included
test_s1 = pd.read_csv(os.path.join(TEST_DIR, 'test_source1.tsv'), sep='\t', usecols=['entity_id', 'business_name', 'country'], dtype=dtype_spec)
test_s1['norm_name'] = test_s1['business_name'].apply(clean_name)

matching_rows = []
candidate_rows = []

for sid, norm, ctry in zip(test_s1['entity_id'], test_s1['norm_name'], test_s1['country']):
    matched = cand_index.get((str(ctry), norm), [])
    pred_str = ','.join(matched[:5]) if matched else ''
    matching_rows.append(f'{sid}\t{pred_str}\n')
    candidate_rows.append(f'{sid}\t{pred_str}\n')

with open(matching_path, 'w', encoding='utf-8') as f:
    f.write('source1_entity_id\tmatched_entity_ids\n')
    f.writelines(matching_rows)

with open(candidate_path, 'w', encoding='utf-8') as f:
    f.write('source1_entity_id\tcandidate_entity_ids\n')
    f.writelines(candidate_rows)

print(f'Saved: {matching_path}')
print(f'Saved: {candidate_path}')

## 12. Validate Submission with Official Validator

In [ ]:
validator_path = '../data/utils/validate_submission.py' if os.path.exists('../data/utils/validate_submission.py') else 'data/utils/validate_submission.py'
print(f'Running validator from {validator_path}...')
# Use python sys.executable to run format validator
import subprocess
cmd = [sys.executable, validator_path, '--matching', os.path.join(out_dir, 'matching_results.tsv'), '--candidate', os.path.join(out_dir, 'candidate_pairs.tsv'), '--test-dir', TEST_DIR]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)